# Profiling / Análisis Exploratorio — Bronze → Stage
### Proyecto: Mortalidad Guatemala / Centroamérica

**Propósito (CRISP-DM · fase *Data Understanding*):** este notebook **no transforma ni escribe nada**.
Solo *perfila* las 5 tablas de la capa **Bronze** para detectar problemas de calidad y, con base en
los hallazgos, justificar las reglas de limpieza que construirán la capa **Stage/Silver**.

**Marco teórico que estructura el notebook (6 dimensiones de calidad de datos):**
Completitud · Unicidad · Validez · Consistencia · Exactitud · Vigencia.

**Notas de plataforma:**
- Corre sobre **Spark Connect / serverless**. Por eso **NO** se fijan credenciales S3 ni se lee de S3
  (eso provocaba el error `fs.s3a.access.key is not available`). Se leen directo las **tablas Delta**
  de Bronze con `spark.table(...)`. El profiling no necesita las credenciales (que además ya fueron rotadas).
- Es **read-only**: ninguna celda hace `write`, `saveAsTable`, `CREATE`, `INSERT` ni `MERGE`.

**Salida:** texto plano (pandas `to_string`) para poder copiar/pegar los resultados fácilmente.


In [0]:
# ── 0. SETUP (read-only, sin S3) ─────────────────────────────────────────────
from pyspark.sql import functions as F
import pandas as pd
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 200)

SCHEMA = "bronze"

# Tablas esperadas en Bronze
TABLAS = ["xlsx_ine", "sav_ine_legacy", "json_oms", "json_worldbank", "gdrive_docs"]
TABLAS_INE = ["xlsx_ine", "sav_ine_legacy"]   # fact de defunciones (núcleo)

# Columnas técnicas de linaje añadidas en la ingesta (no son datos de negocio)
META_COLS = ["_anio", "_archivo_origen", "_fuente"]

# Tokens textuales que el INE usa como "faltante" (centinelas). Se tratan como missing.
UNKNOWN_TOKENS = [
    "", "ignorado", "ignored", "desconocido", "sin informacion", "sin información",
    "no especificado", "n/a", "na", "null", "none", "ninguno", "nan", "<na>", "."
]

def existe_tabla(nombre):
    try:
        spark.table(f"{SCHEMA}.{nombre}")
        return True
    except Exception as e:
        print(f"  [AVISO] no se pudo leer {SCHEMA}.{nombre}: {str(e)[:120]}")
        return False

def rep(titulo, pdf):
    """Imprime un reporte con encabezado, en texto plano y copiable."""
    print("\n" + "=" * 90)
    print(titulo)
    print("=" * 90)
    if isinstance(pdf, pd.DataFrame):
        print(pdf.to_string(index=False))
    else:
        print(pdf)

# Confirmar qué tablas existen realmente
print("Tablas visibles en el esquema bronze:")
spark.sql(f"SHOW TABLES IN {SCHEMA}").show(truncate=False)
TABLAS = [t for t in TABLAS if existe_tabla(t)]
print("\nSe perfilarán:", TABLAS)


Tablas visibles en el esquema bronze:
+--------+--------------+-----------+
|database|tableName     |isTemporary|
+--------+--------------+-----------+
|bronze  |gdrive_docs   |false      |
|bronze  |json_oms      |false      |
|bronze  |json_worldbank|false      |
|bronze  |sav_ine_legacy|false      |
|bronze  |xlsx_ine      |false      |
+--------+--------------+-----------+


Se perfilarán: ['xlsx_ine', 'sav_ine_legacy', 'json_oms', 'json_worldbank', 'gdrive_docs']


In [0]:
# ── 1. INVENTARIO GENERAL (estructura y volumen) ─────────────────────────────
# Dimensión: base para todo. Cuántas filas/columnas y de qué tipo declarado.
filas = []
for t in TABLAS:
    df = spark.table(f"{SCHEMA}.{t}")
    n = df.count()
    cols = df.columns
    filas.append({
        "tabla": t,
        "filas": n,
        "columnas": len(cols),
        "cols_negocio": len([c for c in cols if c not in META_COLS]),
    })
inv = pd.DataFrame(filas)
rep("1. INVENTARIO — volumen por tabla", inv)

print("\nEsquema declarado por tabla (todo string = correcto para Bronze):")
for t in TABLAS:
    print(f"\n--- {SCHEMA}.{t} ---")
    print(", ".join(spark.table(f"{SCHEMA}.{t}").columns))



1. INVENTARIO — volumen por tabla
         tabla  filas  columnas  cols_negocio
      xlsx_ine 674064        30            27
sav_ine_legacy 245167        31            28
      json_oms   1708        27            25
json_worldbank    450        10             8
   gdrive_docs   1837         5             3

Esquema declarado por tabla (todo string = correcto para Bronze):

--- bronze.xlsx_ine ---
Depreg, Mupreg, Mesreg, Añoreg, Depocu, Mupocu, Sexo, Diaocu, Mesocu, Añoocu, Edadif, Perdif, Puedif, Ecidif, Escodif, Ciuodif, Pnadif, Dnadif, Mnadif, Nacdif, Predif, Dredif, Mredif, Caudef, Asist, Ocur, Cerdef, _anio, _archivo_origen, _fuente

--- bronze.sav_ine_legacy ---
Depreg, Mupreg, Mesreg, Añoreg, Depocu, Mupocu, Areag, Sexo, Diaocu, Mesocu, Añoocu, Edadif, Perdif, Puedif, Ecidif, Escodif, Ciuodif, Pnadif, Dnadif, Mnadif, Nacdif, Predif, Dredif, Mredif, Caudef, Asist, Ocur, Cerdef, _anio, _archivo_origen, _fuente

--- bronze.json_oms ---
Id, IndicatorCode, SpatialDimType, SpatialDi

In [0]:
# ── 2. DIFF DE ESQUEMA entre las 2 tablas INE (consistencia estructural) ──────
# Dimensión: Consistencia. ¿Se pueden unificar 2015-2017 y 2018-2024 en una sola
# tabla Stage? Esto responde la decisión "unificar vs separar".
if all(t in TABLAS for t in TABLAS_INE):
    c_xlsx = spark.table(f"{SCHEMA}.xlsx_ine").columns
    c_leg  = spark.table(f"{SCHEMA}.sav_ine_legacy").columns
    s_xlsx, s_leg = set(c_xlsx), set(c_leg)

    diff = pd.DataFrame({
        "situacion": ["solo en xlsx_ine (2018-2024)",
                      "solo en sav_ine_legacy (2015-2017)",
                      "en ambas"],
        "columnas": [", ".join(sorted(s_xlsx - s_leg)) or "(ninguna)",
                     ", ".join(sorted(s_leg - s_xlsx)) or "(ninguna)",
                     f"{len(s_xlsx & s_leg)} columnas comunes"],
    })
    rep("2. DIFF DE ESQUEMA INE (clave para unificar Stage)", diff)
    print("\nLectura: si la única diferencia es 'Areag', la unión es viable")
    print("añadiendo Areag=NULL al periodo que no la trae (o conservándola donde exista).")
else:
    print("No están ambas tablas INE; se omite el diff.")



2. DIFF DE ESQUEMA INE (clave para unificar Stage)
                         situacion            columnas
      solo en xlsx_ine (2018-2024)           (ninguna)
solo en sav_ine_legacy (2015-2017)               Areag
                          en ambas 30 columnas comunes

Lectura: si la única diferencia es 'Areag', la unión es viable
añadiendo Areag=NULL al periodo que no la trae (o conservándola donde exista).


In [0]:
# ── 3. COMPLETITUD (nulos + centinelas de 'faltante') ────────────────────────
# Dimensión: Completitud. Cuenta, por columna, valores faltantes reales:
#   NULL  |  cadena vacía/espacios  |  token textual de desconocido ("Ignorado", etc.)
def perfil_completitud(t):
    df = spark.table(f"{SCHEMA}.{t}")
    total = df.count()
    cols = df.columns
    exprs = []
    for c in cols:
        col = df[c]
        norm = F.lower(F.trim(col))
        is_missing = col.isNull() | norm.isin(UNKNOWN_TOKENS)
        exprs.append(F.sum(is_missing.cast("int")).alias(c))
    row = df.agg(*exprs).collect()[0].asDict()
    out = pd.DataFrame(
        [{"columna": c, "faltantes": int(row[c]),
          "pct_faltante": round(100*row[c]/total, 2) if total else None}
         for c in cols]
    ).sort_values("pct_faltante", ascending=False)
    return total, out

for t in TABLAS:
    total, out = perfil_completitud(t)
    rep(f"3. COMPLETITUD — {SCHEMA}.{t}  (filas={total:,})", out)



3. COMPLETITUD — bronze.xlsx_ine  (filas=674,064)
        columna  faltantes  pct_faltante
        Ciuodif      99593         14.78
        Escodif      99593         14.78
         Pnadif          0          0.00
_archivo_origen          0          0.00
          _anio          0          0.00
         Cerdef          0          0.00
           Ocur          0          0.00
          Asist          0          0.00
         Caudef          0          0.00
         Mredif          0          0.00
         Dredif          0          0.00
         Predif          0          0.00
         Nacdif          0          0.00
         Mnadif          0          0.00
         Dnadif          0          0.00
         Depreg          0          0.00
         Mupreg          0          0.00
         Ecidif          0          0.00
         Puedif          0          0.00
         Perdif          0          0.00
         Edadif          0          0.00
         Añoocu          0          0.00
      

In [0]:
# ── 4. UNICIDAD (duplicados) ─────────────────────────────────────────────────
# Dimensión: Unicidad. Duplicados de fila completa y duplicados ignorando metadata
# de linaje (porque _archivo_origen puede diferir aunque el registro sea el mismo).
filas = []
for t in TABLAS:
    df = spark.table(f"{SCHEMA}.{t}")
    total = df.count()
    dups_full = total - df.dropDuplicates().count()
    cols_neg = [c for c in df.columns if c not in META_COLS]
    dups_neg = total - df.select(*cols_neg).dropDuplicates().count()
    filas.append({
        "tabla": t,
        "filas": total,
        "dups_fila_completa": dups_full,
        "dups_sin_metadata": dups_neg,
        "pct_dups_sin_meta": round(100*dups_neg/total, 2) if total else None,
    })
rep("4. UNICIDAD — duplicados por tabla", pd.DataFrame(filas))
print("\nNota: 'dups_sin_metadata' es el indicador relevante para deduplicar en Stage.")
print("Un valor alto en INE puede ser real (varias muertes con mismos atributos) —")
print("NO deduplicar a ciegas; primero definir la llave de negocio del registro.")



4. UNICIDAD — duplicados por tabla
         tabla  filas  dups_fila_completa  dups_sin_metadata  pct_dups_sin_meta
      xlsx_ine 674064                 167                167               0.02
sav_ine_legacy 245167                  79                 79               0.03
      json_oms   1708                   0                  0               0.00
json_worldbank    450                   0                  0               0.00
   gdrive_docs   1837                1193               1193              64.94

Nota: 'dups_sin_metadata' es el indicador relevante para deduplicar en Stage.
Un valor alto en INE puede ser real (varias muertes con mismos atributos) —
NO deduplicar a ciegas; primero definir la llave de negocio del registro.


In [0]:
# ── 5. VALIDEZ — dominios y rangos (tablas INE) ──────────────────────────────
# Dimensión: Validez. ¿Los valores respetan el dominio/rango esperado?
def to_int(c):
    # robusto a "2015.0" (artefacto de parquet->str) y a texto no numérico -> NULL
    return F.col(f"`{c}`").cast("double").cast("int")

def distribucion(t, c, n=25):
    df = spark.table(f"{SCHEMA}.{t}")
    if c not in df.columns:
        return None
    d = (df.groupBy(c).count().orderBy(F.desc("count")).limit(n).toPandas())
    return d

for t in [x for x in TABLAS_INE if x in TABLAS]:
    print("\n" + "#" * 90)
    print(f"# VALIDEZ — {SCHEMA}.{t}")
    print("#" * 90)
    df = spark.table(f"{SCHEMA}.{t}")

    # 5.1 Sexo: distribución de códigos
    d = distribucion(t, "Sexo")
    if d is not None:
        rep(f"5.1 Sexo (distribución de códigos crudos) — {t}", d)

    # 5.2 Mes de ocurrencia y registro: deben estar en 1..12
    for mescol in ["Mesocu", "Mesreg"]:
        if mescol in df.columns:
            m = to_int(mescol)
            fuera = df.filter(~(m.between(1, 12)) | m.isNull()).count()
            rep(f"5.2 {mescol} fuera de 1..12 o no numérico — {t}",
                pd.DataFrame([{"columna": mescol, "violaciones": fuera}]))

    # 5.3 Día de ocurrencia: 1..31
    if "Diaocu" in df.columns:
        dd = to_int("Diaocu")
        fuera = df.filter(~(dd.between(1, 31)) | dd.isNull()).count()
        rep(f"5.3 Diaocu fuera de 1..31 o no numérico — {t}",
            pd.DataFrame([{"columna": "Diaocu", "violaciones": fuera}]))

    # 5.4 Edad del difunto: rango plausible 0..120 (solo como tamiz)
    if "Edadif" in df.columns:
        e = to_int("Edadif")
        no_num = df.filter(e.isNull() & df["Edadif"].isNotNull()).count()
        fuera = df.filter(e.isNotNull() & ~e.between(0, 120)).count()
        rep(f"5.4 Edadif — calidad numérica — {t}", pd.DataFrame([
            {"chequeo": "no convertible a número", "conteo": no_num},
            {"chequeo": "fuera de 0..120", "conteo": fuera},
        ]))
        # nota: Edadif suele venir con un código de unidad (Perdif: años/meses/días)
        if "Perdif" in df.columns:
            rep(f"5.4b Perdif (unidad de edad) distribución — {t}",
                distribucion(t, "Perdif"))

    # 5.5 Caudef: causa de muerte. ¿Parece código CIE-10? (letra + 2 dígitos ...)
    if "Caudef" in df.columns:
        patron = r"^[A-Z][0-9]{2}.*$"
        ok = df.filter(F.upper(F.trim(df["Caudef"])).rlike(patron)).count()
        tot = df.count()
        rep(f"5.5 Caudef con formato tipo CIE-10 (Lnn...) — {t}", pd.DataFrame([{
            "con_patron_CIE10": ok, "total": tot,
            "pct": round(100*ok/tot, 2) if tot else None}]))
        rep(f"5.5b Caudef — top valores — {t}", distribucion(t, "Caudef", 15))



##########################################################################################
# VALIDEZ — bronze.xlsx_ine
##########################################################################################

5.1 Sexo (distribución de códigos crudos) — xlsx_ine
Sexo  count
   1 374733
   2 299331

5.2 Mesocu fuera de 1..12 o no numérico — xlsx_ine
columna  violaciones
 Mesocu            0

5.2 Mesreg fuera de 1..12 o no numérico — xlsx_ine
columna  violaciones
 Mesreg            0

5.3 Diaocu fuera de 1..31 o no numérico — xlsx_ine
columna  violaciones
 Diaocu            0

5.4 Edadif — calidad numérica — xlsx_ine
                chequeo  conteo
no convertible a número       0
        fuera de 0..120    1997

5.4b Perdif (unidad de edad) distribución — xlsx_ine
Perdif  count
     3 626139
     1  24168
     2  21760
     9   1997

5.5 Caudef con formato tipo CIE-10 (Lnn...) — xlsx_ine
 con_patron_CIE10  total   pct
           674064 674064 100.0

5.5b Caudef — top valores — xlsx_ine

In [0]:
# ── 6. CONSISTENCIA — reglas cruzadas (tablas INE) ───────────────────────────
# Dimensión: Consistencia. Relaciones lógicas entre columnas.
for t in [x for x in TABLAS_INE if x in TABLAS]:
    df = spark.table(f"{SCHEMA}.{t}")
    chequeos = []

    # 6.1 Año de ocurrencia no puede ser posterior al año de registro
    if "Añoocu" in df.columns and "Añoreg" in df.columns:
        ao = F.col("`Añoocu`").cast("double").cast("int")
        ar = F.col("`Añoreg`").cast("double").cast("int")
        viol = df.filter(ao.isNotNull() & ar.isNotNull() & (ao > ar)).count()
        chequeos.append({"regla": "Añoocu <= Añoreg", "violaciones": viol})

    # 6.2 _anio (año del archivo) debe coincidir con Añoreg
    if "_anio" in df.columns and "Añoreg" in df.columns:
        a1 = F.col("_anio").cast("double").cast("int")
        a2 = F.col("`Añoreg`").cast("double").cast("int")
        mis = df.filter(a1.isNotNull() & a2.isNotNull() & (a1 != a2)).count()
        chequeos.append({"regla": "_anio == Añoreg (año de registro)", "violaciones": mis})

    # 6.3 Rango de años de ocurrencia observado (vigencia / outliers temporales)
    if "Añoocu" in df.columns:
        ao = F.col("`Añoocu`").cast("double").cast("int")
        r = df.select(F.min(ao).alias("min"), F.max(ao).alias("max")).collect()[0]
        chequeos.append({"regla": f"rango Añoocu observado", "violaciones": f"{r['min']}..{r['max']}"})

    rep(f"6. CONSISTENCIA — {SCHEMA}.{t}", pd.DataFrame(chequeos))



6. CONSISTENCIA — bronze.xlsx_ine
                            regla violaciones
                 Añoocu <= Añoreg           0
_anio == Añoreg (año de registro)       23971
           rango Añoocu observado  2018..2024

6. CONSISTENCIA — bronze.sav_ine_legacy
                            regla violaciones
                 Añoocu <= Añoreg           0
_anio == Añoreg (año de registro)        6043
           rango Añoocu observado  2015..2017


In [0]:
# ── 7. CARDINALIDAD / DISTRIBUCIÓN (tablas INE) ──────────────────────────────
# Dimensión: apoya Validez+Exactitud. Cardinalidad aproximada por columna:
# detecta columnas casi-constantes, identificadores y categóricas a estandarizar.
for t in [x for x in TABLAS_INE if x in TABLAS]:
    df = spark.table(f"{SCHEMA}.{t}")
    cols = [c for c in df.columns if c not in META_COLS]
    exprs = [F.approx_count_distinct(df[c]).alias(c) for c in cols]
    row = df.agg(*exprs).collect()[0].asDict()
    card = pd.DataFrame([{"columna": c, "valores_distintos_aprox": int(row[c])}
                         for c in cols]).sort_values("valores_distintos_aprox")
    rep(f"7. CARDINALIDAD aprox — {SCHEMA}.{t}", card)



7. CARDINALIDAD aprox — bronze.xlsx_ine
columna  valores_distintos_aprox
   Sexo                        2
 Ecidif                        4
 Perdif                        4
 Cerdef                        4
  Asist                        6
 Puedif                        6
 Añoocu                        7
Escodif                        7
 Añoreg                        8
   Ocur                        8
 Mesreg                       12
 Mesocu                       12
 Depocu                       22
 Depreg                       22
 Predif                       24
 Dnadif                       25
 Dredif                       25
 Diaocu                       32
Ciuodif                       44
 Pnadif                       85
 Nacdif                       85
 Edadif                      125
 Mupocu                      324
 Mupreg                      324
 Mnadif                      328
 Mredif                      328
 Caudef                     2692

7. CARDINALIDAD aprox — bronze.sav

In [0]:
# ── 8. PERFILADO — bronze.json_oms (datos de referencia OMS) ─────────────────
# Tabla de referencia (esperanza de vida, esp. de vida saludable, mortalidad <5).
if "json_oms" in TABLAS:
    df = spark.table(f"{SCHEMA}.json_oms")
    total = df.count()
    print(f"json_oms filas = {total:,}")

    for c in ["IndicatorCode", "SpatialDim", "ParentLocation", "Dim1", "TimeDim"]:
        if c in df.columns:
            rep(f"8. OMS — distribución de {c}",
                df.groupBy(c).count().orderBy(F.desc("count")).limit(25).toPandas())

    # Completitud del valor numérico
    if "NumericValue" in df.columns:
        nv = F.col("NumericValue")
        miss = df.filter(nv.isNull() | (F.lower(F.trim(nv)).isin(UNKNOWN_TOKENS))).count()
        rep("8. OMS — completitud de NumericValue", pd.DataFrame([{
            "faltantes": miss, "total": total,
            "pct": round(100*miss/total, 2) if total else None}]))
else:
    print("json_oms no disponible.")


json_oms filas = 1,708

8. OMS — distribución de IndicatorCode
 IndicatorCode  count
MDG_0000000001   1048
 WHOSIS_000001    330
 WHOSIS_000002    330

8. OMS — distribución de SpatialDim
SpatialDim  count
       CRI    355
       PAN    345
       SLV    339
       HND    336
       GTM    333

8. OMS — distribución de ParentLocation
ParentLocation  count
      Americas   1708

8. OMS — distribución de Dim1
    Dim1  count
SEX_BTSX    570
 SEX_MLE    569
SEX_FMLE    569

8. OMS — distribución de TimeDim
TimeDim  count
   2009     45
   2004     45
   2010     45
   2020     45
   2007     45
   2011     45
   2000     45
   2013     45
   2021     45
   2017     45
   2006     45
   2001     45
   2014     45
   2008     45
   2002     45
   2012     45
   2018     45
   2003     45
   2015     45
   2016     45
   2005     45
   2019     45
   1958     15
   1966     15
   1977     15

8. OMS — completitud de NumericValue
 faltantes  total  pct
         0   1708  0.0


In [0]:
# ── 9. PERFILADO — bronze.json_worldbank (datos de referencia Banco Mundial) ──
if "json_worldbank" in TABLAS:
    df = spark.table(f"{SCHEMA}.json_worldbank")
    total = df.count()
    print(f"json_worldbank filas = {total:,}")

    # 'indicator' y 'country' suelen venir como dict serializado en string -> revisar
    rep("9. WB — muestra de filas", df.limit(8).toPandas())

    for c in ["countryiso3code", "date"]:
        if c in df.columns:
            rep(f"9. WB — distribución de {c}",
                df.groupBy(c).count().orderBy(F.desc("count")).limit(30).toPandas())

    if "value" in df.columns:
        v = F.col("value")
        miss = df.filter(v.isNull() | F.lower(F.trim(v)).isin(UNKNOWN_TOKENS)).count()
        rep("9. WB — completitud de value", pd.DataFrame([{
            "faltantes": miss, "total": total,
            "pct": round(100*miss/total, 2) if total else None}]))
else:
    print("json_worldbank no disponible.")


json_worldbank filas = 450

9. WB — muestra de filas
                                                                indicator                             country countryiso3code date value unit obs_status decimal                                                 _archivo_origen   _fuente
{'id': 'SP.DYN.CDRT.IN', 'value': 'Death rate, crude (per 1,000 people)'} {'id': 'CR', 'value': 'Costa Rica'}             CRI 2024 5.566                       0 raw/centroamerica/worldbank_crude_death_rate_centroamerica.json WORLDBANK
{'id': 'SP.DYN.CDRT.IN', 'value': 'Death rate, crude (per 1,000 people)'} {'id': 'CR', 'value': 'Costa Rica'}             CRI 2023 5.457                       0 raw/centroamerica/worldbank_crude_death_rate_centroamerica.json WORLDBANK
{'id': 'SP.DYN.CDRT.IN', 'value': 'Death rate, crude (per 1,000 people)'} {'id': 'CR', 'value': 'Costa Rica'}             CRI 2022 6.106                       0 raw/centroamerica/worldbank_crude_death_rate_centroamerica.json WORLDBANK
{'id': 

In [0]:
# ── 10. PERFILADO — bronze.gdrive_docs (diccionario de variables) ────────────
# Metadata. Se usará como REFERENCIA DE VALIDACIÓN (códigos huérfanos) y, en Gold,
# como base de las dimensiones. Aquí solo inspeccionamos su estructura.
if "gdrive_docs" in TABLAS:
    df = spark.table(f"{SCHEMA}.gdrive_docs")
    print(f"gdrive_docs filas = {df.count():,}")
    print("Columnas:", df.columns)
    rep("10. DICCIONARIO — primeras 40 filas", df.limit(40).toPandas())
else:
    print("gdrive_docs no disponible.")


gdrive_docs filas = 1,837
Columnas: ['Valores_de_las_variables_defunciones', 'Unnamed:_1', 'Unnamed:_2', '_archivo_origen', '_fuente']

10. DICCIONARIO — primeras 40 filas
Valores_de_las_variables_defunciones Unnamed:_1             Unnamed:_2                             _archivo_origen      _fuente
                               Valor     Código               Etiqueta raw/gdrive/diccionario_defunciones_ine.xlsx GOOGLE_DRIVE
            Departamento de registro          1              Guatemala raw/gdrive/diccionario_defunciones_ine.xlsx GOOGLE_DRIVE
                                None          2            El Progreso raw/gdrive/diccionario_defunciones_ine.xlsx GOOGLE_DRIVE
                                None          3           Sacatepéquez raw/gdrive/diccionario_defunciones_ine.xlsx GOOGLE_DRIVE
                                None          4          Chimaltenango raw/gdrive/diccionario_defunciones_ine.xlsx GOOGLE_DRIVE
                                None          5             

In [0]:
# ── 11. RESUMEN CONSOLIDADO ──────────────────────────────────────────────────
# Una sola tabla con los indicadores macro por tabla, para decidir próximos pasos.
filas = []
for t in TABLAS:
    df = spark.table(f"{SCHEMA}.{t}")
    total = df.count()
    cols = df.columns
    cols_neg = [c for c in cols if c not in META_COLS]
    dups_neg = total - df.select(*cols_neg).dropDuplicates().count() if cols_neg else 0
    filas.append({
        "tabla": t, "filas": total, "columnas": len(cols),
        "dups_sin_metadata": dups_neg,
    })
rep("11. RESUMEN CONSOLIDADO", pd.DataFrame(filas))
print("\nFin del profiling. Comparte esta salida para evaluar si es suficiente")
print("para definir las reglas de Stage (o si hace falta una segunda pasada).")



11. RESUMEN CONSOLIDADO
         tabla  filas  columnas  dups_sin_metadata
      xlsx_ine 674064        30                167
sav_ine_legacy 245167        31                 79
      json_oms   1708        27                  0
json_worldbank    450        10                  0
   gdrive_docs   1837         5               1193

Fin del profiling. Comparte esta salida para evaluar si es suficiente
para definir las reglas de Stage (o si hace falta una segunda pasada).
